# Pretrained Vision Transformer Walkthrough

This notebook mirrors the Vision Transformers homework flow. The Python script is the experiment engine; the notebook keeps the dataset, baseline, ablation, mistake inspection, and final-test guard visible in one place.

Use the quick path to verify the environment. Use script-written artifacts, fixed validation evidence, and one reserved final-test run for any result you report.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

In [ ]:
from pathlib import Path
import os
import sys


def find_chapter_dir(script_name: str, chapter_name: str) -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


CHAPTER_DIR = find_chapter_dir("pretrained_vit_experiment.py", "chapter_vision_transformers")
os.chdir(CHAPTER_DIR)
sys.path.insert(0, str(CHAPTER_DIR))

from pretrained_vit_experiment import make_args, run_experiment

RUN_QUICK_CHECK = True
RUN_FULL_BASELINE = False
RUN_ABLATION = False
RUN_FINAL_TEST = False
FINAL_RECIPE = "lora"  # choose "head" or "lora" after validation evidence

print("Chapter directory:", CHAPTER_DIR)

## 1: define the dataset and split

Start with the synthetic quick check to verify the environment. For the real homework run, switch to Food-101 and record the split rule in the report. If hardware is limited, use a controlled Food-101 subset or the CIFAR-10 fallback and state the limitation.

In [ ]:
quick_args = make_args(
    quick=True,
    output_dir=Path("runs/notebook-vit-quick"),
)

baseline_args = make_args(
    dataset="food101",
    download=True,
    data_root=Path("~/.cache/torch/datasets").expanduser(),
    checkpoint="google/vit-base-patch16-224-in21k",
    image_size=224,
    resize_size=256,
    batch_size=16,
    trainable_mode="head",
    epochs=3,
    learning_rate=3e-4,
    weight_decay=0.01,
    valid_pct=0.2,
    seed=1234,
    output_dir=Path("runs/vit-food101-head"),
)

print("Quick dataset:", quick_args.dataset)
print("Baseline dataset:", baseline_args.dataset)
print("Validation pct:", baseline_args.valid_pct)
print("Checkpoint:", baseline_args.checkpoint)

## 2: run the pretrained baseline

The quick run should not be reported as a model result. It only checks that the pipeline can train, evaluate, and write artifacts. The full baseline freezes the pretrained body and trains the classification head, giving the comparison point for later ablations.

In [ ]:
quick_result = None
if RUN_QUICK_CHECK:
    quick_result = run_experiment(quick_args)

baseline_result = None
if RUN_FULL_BASELINE:
    baseline_result = run_experiment(baseline_args)

## 3: run one controlled ablation

Change one main factor. This ablation keeps the data, split, checkpoint, image size, and epoch count fixed, but switches from a head-only update to LoRA adapters on the attention query and value projections.

In [ ]:
ablation_args = make_args(
    dataset="food101",
    download=False,
    data_root=baseline_args.data_root,
    checkpoint=baseline_args.checkpoint,
    image_size=baseline_args.image_size,
    resize_size=baseline_args.resize_size,
    batch_size=baseline_args.batch_size,
    trainable_mode="lora",
    epochs=baseline_args.epochs,
    learning_rate=1e-3,
    weight_decay=baseline_args.weight_decay,
    valid_pct=baseline_args.valid_pct,
    lora_r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    lora_target_modules=["query", "value"],
    seed=baseline_args.seed,
    output_dir=Path("runs/vit-food101-lora"),
)

ablation_result = None
if RUN_ABLATION:
    ablation_result = run_experiment(ablation_args)

## 4: inspect mistakes and report cost

Each run writes a result summary and validation mistake table. Inspect mistakes before deciding whether the ablation is worth its extra cost, and compare accuracy with latency, parameter count, and artifact metadata rather than using accuracy alone.

In [ ]:
import csv

result_to_inspect = baseline_result or quick_result
if result_to_inspect is not None:
    print(result_to_inspect["metrics"])
    mistake_path = Path(result_to_inspect["artifacts"]["validation_prediction_examples"])
    with mistake_path.open(encoding="utf-8") as handle:
        mistakes = list(csv.DictReader(handle))
    mistakes[:10]

## 5: final test measurement

Set `RUN_FINAL_TEST = True` only after choosing the recipe from validation evidence. Do not use this cell to pick between baseline and ablation runs. The final-test artifact should be a one-time confirmation of the selected recipe.

In [ ]:
selected_args = ablation_args if FINAL_RECIPE == "lora" else baseline_args
final_args = make_args(**vars(selected_args))
final_args.evaluate_test = True
final_args.output_dir = Path("runs/vit-food101-final-test")

final_result = None
if RUN_FINAL_TEST:
    final_result = run_experiment(final_args)
    print(final_result["metrics"]["test"])

## What To Report

A reportable ViT comparison should name the dataset split, checkpoint, image size, frozen or adapted parameter surface, seed, epoch budget, hardware, validation metric, and final selected test result. Include at least one mistake pattern from the validation table so the comparison is tied to model behavior, not just aggregate scores.

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.